#Deep Learning Based DDoS Attack Detection

Import Libraries and Models

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

Load Dataset and Preprocessing

In [ ]:
    df = pd.read_csv("/content/Portmap.csv")
    print("Data shape:", df.shape)
    print("Class distribution:\n", df['Label'].value_counts())
    # Drop non-numeric or irrelevant columns
    drop_cols = ['SourceIP', 'DestinationIP', 'Protocol', 'Timestamp', 'Unnamed: 0', 'Unnamed0']
    df.drop([col for col in drop_cols if col in df.columns], axis=1, inplace=True)

    normal_terms = ['normal', 'benign']
    df['Label'] = df['Label'].astype(str).str.lower().str.strip()
    df['Label'] = df['Label'].apply(lambda x: 0 if x in normal_terms else 1)

    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.fillna(0, inplace=True)

    for col in df.columns:
        if df[col].dtype in ['float64', 'int64']:
            df[col] = df[col].clip(-1e10, 1e10)
        elif df[col].dtype == 'object':
            try:
                df[col] = pd.to_numeric(df[col])
            except:
                df.drop(col, axis=1, inplace=True)


/tmp/ipython-input-4266402483.py:1: DtypeWarning: Columns (85) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/content/Portmap.csv")


Data shape: (191694, 88)
Class distribution:
 Label
Portmap    186960
BENIGN       4734
Name: count, dtype: int64


Splitting Dataset (Training and Testing)

In [ ]:
X = df.drop('Label', axis=1).values
y = df['Label'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train = np.nan_to_num(X_train, nan=0, posinf=1e10, neginf=-1e10)
X_test = np.nan_to_num(X_test, nan=0, posinf=1e10, neginf=-1e10)

X_train = X_train.reshape((X_train.shape[0], 1, X_train.shape[1]))
X_test = X_test.reshape((X_test.shape[0], 1, X_test.shape[1]))

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (153355, 1, 81)
Test shape: (38339, 1, 81)


LSTM Model

In [ ]:
def lstm_model(input_shape):
    model = Sequential()
    model.add(LSTM(64, input_shape=input_shape, return_sequences=True))
    model.add(Dropout(0.3))
    model.add(LSTM(32))
    model.add(Dropout(0.3))
    model.add(Dense(1, activation='sigmoid'))

    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

    return model

model = lstm_model((X_train.shape[1], X_train.shape[2]))

history = model.fit(X_train, y_train, validation_split=0.2, epochs=3)

test_pred = (model.predict(X_test) > 0.5).astype(int)
test_accuracy = accuracy_score(y_test, test_pred)
print("LSTM Model Accuracy: ", test_accuracy)
print(classification_report(y_test, test_pred))

Epoch 1/3
3834/3834 ━━━━━━━━━━━━━━━━━━━━ 30s 7ms/step - accuracy: 0.9948 - loss: 0.0581 - val_accuracy: 0.9986 - val_loss: 0.0042
Epoch 2/3
3834/3834 ━━━━━━━━━━━━━━━━━━━━ 28s 7ms/step - accuracy: 0.9991 - loss: 0.0038 - val_accuracy: 0.9992 - val_loss: 0.0029
Epoch 3/3
3834/3834 ━━━━━━━━━━━━━━━━━━━━ 28s 7ms/step - accuracy: 0.9996 - loss: 0.0022 - val_accuracy: 0.9997 - val_loss: 0.0016
1199/1199 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step
LSTM Model Accuracy:  0.9998435013954459
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       930
           1       1.00      1.00      1.00     37409

    accuracy                           1.00     38339
   macro avg       1.00      1.00      1.00     38339
weighted avg       1.00      1.00      1.00     38339



In [ ]:
model.save("lstm_ddos_model.h5")


In [ ]:
import joblib

joblib.dump(scaler, "scaler.pkl")


['scaler.pkl']